# Build RAG Index on Kaggle GPU

1. Upload `records.jsonl` as Kaggle dataset `guap-raw`
2. Enable **GPU T4 x2**
3. Run cells **in order**
4. Download `rag_index.tar.gz` from Output

In [ ]:
# Cell 1 — Install deps (run ONCE, then restart runtime)
!pip install -q transformers==4.51.0 sentence-transformers==5.1.1 einops
!pip install -q faiss-cpu

### ⚠️ After cell above finishes: Runtime → Restart session, then run cells below

In [ ]:
# Cell 2 — Clone repo + setup paths
import transformers, sys
print(f'transformers=={transformers.__version__}')

!rm -rf /tmp/llm-speaker-core
!GIT_LFS_SKIP_SMUDGE=1 GIT_TERMINAL_PROMPT=0 git clone https://github.com/chudinovAI/llm-speaker-core.git /tmp/llm-speaker-core
sys.path.insert(0, '/tmp/llm-speaker-core/src')

from pathlib import Path
RAW_CANDIDATES = [
    Path('/kaggle/input/guap-raw/records.jsonl'),
    Path('/tmp/llm-speaker-core/data/raw/cloudflare/latest/records.jsonl'),
]
RAW_RECORDS = next((p for p in RAW_CANDIDATES if p.exists()), None)
assert RAW_RECORDS, f'records.jsonl not found: {RAW_CANDIDATES}'
print(f'Data: {RAW_RECORDS}')

OUT = Path('/kaggle/working/rag_output')
for d in ['indexes/bm25', 'indexes/faiss', 'normalized']:
    (OUT / d).mkdir(parents=True, exist_ok=True)
print('OK')

In [ ]:
# Cell 3 — Normalize + chunk + BM25
from llm_speaker_core.ingest.normalize import (
    load_cloudflare_documents, dedupe_documents, build_chunk_corpus,
    write_documents, write_chunks,
)
from llm_speaker_core.retrieval.lexical import LexicalIndex

documents = dedupe_documents(load_cloudflare_documents(RAW_RECORDS))
chunks = build_chunk_corpus(documents)
write_documents(OUT / 'normalized/documents.jsonl', documents)
write_chunks(OUT / 'normalized/chunks.jsonl', chunks)
print(f'Documents: {len(documents)}, Chunks: {len(chunks)}')

lexical = LexicalIndex.build(chunks)
lexical.save(OUT / 'indexes/bm25/index.json')
print('BM25 done')

In [ ]:
# Cell 4 — Load embedding model (AutoModel, NOT SentenceTransformer)
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'ai-sage/Giga-Embeddings-instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME, trust_remote_code=True, torch_dtype=torch.bfloat16,
)
model.eval().cuda()
print(f'Model loaded on {next(model.parameters()).device}')

In [ ]:
# Cell 5 — Encode all chunks → dense vectors
import numpy as np

def encode_texts(texts, batch_size=8):
    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch, padding=True, truncation=True,
            max_length=4096, return_tensors='pt',
        )
        enc = {k: v.cuda() for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc, return_embeddings=True)
        if isinstance(out, torch.Tensor):
            emb = out
        elif hasattr(out, 'last_hidden_state'):
            emb = out.last_hidden_state[:, 0]
        else:
            emb = out[0][:, 0]
        all_emb.append(emb.cpu().float().numpy())
        if (i // batch_size) % 10 == 0:
            print(f'  {i}/{len(texts)}', flush=True)
    result = np.vstack(all_emb)
    norms = np.linalg.norm(result, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return result / norms

print(f'Encoding {len(chunks)} chunks...')
vectors = encode_texts([c.text for c in chunks])
print(f'Done: {vectors.shape}')

In [ ]:
# Cell 6 — Save FAISS index + manifest + archive
import json, datetime, tarfile, faiss
from llm_speaker_core.retrieval.schemas import IndexManifest
from dataclasses import asdict

vpath = OUT / 'indexes/faiss/index.vectors.npy'
fpath = OUT / 'indexes/faiss/index.faiss'
jpath = OUT / 'indexes/faiss/index.json'

np.save(vpath, vectors.astype(np.float32))
jpath.write_text(json.dumps({
    'model_name': MODEL_NAME, 'available': True,
    'chunks': [c.__dict__ for c in chunks],
}, ensure_ascii=False), encoding='utf-8')

ix = faiss.IndexFlatIP(vectors.shape[1])
ix.add(vectors.astype(np.float32))
faiss.write_index(ix, str(fpath))

manifest = IndexManifest(
    version='hybrid-rag-v3',
    corpus_checksum=str(abs(hash(''.join(c.content_hash for c in chunks))) % (10**16)),
    lexical_path='indexes/bm25/index.json',
    dense_path='indexes/faiss/index.json',
    reranker_model='BAAI/bge-reranker-v2-m3',
    embedding_model=MODEL_NAME,
    built_at=datetime.datetime.utcnow().isoformat() + 'Z',
    doc_count=len({c.doc_id for c in chunks}),
    chunk_count=len(chunks),
    metadata={'storage': 'faiss+jsonl', 'dense_available': True, 'reranker_available': True},
)
(OUT / 'index_manifest.json').write_text(
    json.dumps(asdict(manifest), ensure_ascii=False, indent=2), encoding='utf-8',
)

archive = '/kaggle/working/rag_index.tar.gz'
with tarfile.open(archive, 'w:gz') as tar:
    for src, arc in [
        (OUT/'index_manifest.json',          'data/index_manifest.json'),
        (OUT/'indexes/bm25/index.json',      'data/indexes/bm25/index.json'),
        (jpath,                              'data/indexes/faiss/index.json'),
        (vpath,                              'data/indexes/faiss/index.vectors.npy'),
        (fpath,                              'data/indexes/faiss/index.faiss'),
        (OUT/'normalized/documents.jsonl',   'data/normalized/documents.jsonl'),
        (OUT/'normalized/chunks.jsonl',      'data/normalized/chunks.jsonl'),
    ]:
        tar.add(src, arcname=arc)

mb = Path(archive).stat().st_size / 1024 / 1024
print(f'Archive: {archive} ({mb:.1f} MB)')
print('On Mac: cd llm-speaker-core && tar xzf ~/Downloads/rag_index.tar.gz')